# Modeling tuning


## Setup

In [1]:
try:
    from synthefy_nori import NoriRegressor
    NORI_AVAILABLE = True
except ImportError:
    NORI_AVAILABLE = False

In [2]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from skopt import BayesSearchCV
from skopt.space import Integer, Real

import helpers  # noqa: F401  (puts scripts/ on sys.path, loads .env)
from helpers.data import load_historical, load_params
from helpers.modeling import RecencyXGB, holdout_report, resolve_features, time_holdout_split
from features import FEATURE_COLUMNS
from model import walk_forward_evaluate


In [3]:
USE_LIVE_DATA = False  # set True to pull real Strava activities + Open-Meteo forecast via .env secrets

historical = load_historical(USE_LIVE_DATA)
params = load_params()


In [ ]:
historical[historical.days_since_last_hard.isna()]

,as_of_date,target_date,acute_load_7,chronic_load_28,atl_ctl_ratio,days_since_last_hard,streak_length,trained_today,trained_hard_today,trained_days_2,...,forecast_wind_speed,forecast_temp_high_vs_seasonal,forecast_temp_low_vs_seasonal,forecast_precip_probability_vs_seasonal,forecast_wind_speed_vs_seasonal,forecast_precip_morning,forecast_precip_midday,forecast_precip_evening,will_train_tomorrow,next_day_relative_effort
0,2023-10-10,2023-10-11,0.00000,0.000000,0.000000,999,0,0,0,0.0,...,12.3,-1.495082,0.646230,29.371585,0.028852,100.0,100.000000,20.0,0,0.0
1,2023-10-11,2023-10-12,0.00000,0.000000,0.000000,999,0,0,0,0.0,...,6.8,2.221967,-1.387869,-29.262295,-5.510820,0.0,0.000000,0.0,0,0.0
2,2023-10-12,2023-10-13,0.00000,0.000000,0.000000,999,0,0,0,0.0,...,19.2,4.912131,-1.728197,-16.898907,6.863934,0.0,0.000000,0.0,0,0.0
3,2023-10-13,2023-10-14,0.00000,0.000000,0.000000,999,0,0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
4,2023-10-14,2023-10-15,0.00000,0.000000,0.000000,999,0,0,0,0.0,...,8.4,2.054426,2.038689,-9.057377,-4.107541,0.0,14.285714,80.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,2024-06-14,2024-06-15,0.00000,0.000000,0.000000,999,0,0,0,0.0,...,18.7,-8.510667,-2.231000,57.250000,7.037333,100.0,100.000000,100.0,0,0.0
249,2024-06-15,2024-06-16,0.00000,0.000000,0.000000,999,0,0,0,0.0,...,13.4,-4.710033,-3.413378,15.914158,1.781271,50.0,57.142857,40.0,1,6.0
250,2024-06-16,2024-06-17,1.50000,0.413793,3.625000,999,1,1,0,1.0,...,11.5,-5.774247,-1.808361,40.914158,-0.064548,75.0,71.428571,80.0,0,0.0
251,2024-06-17,2024-06-18,1.12500,0.385256,2.920139,999,0,0,0,1.0,...,9.1,-3.361538,-2.174247,-13.154961,-2.472241,0.0,14.285714,0.0,1,1.0


In [ ]:
selected_features = params["model"].get("features", FEATURE_COLUMNS)
features = resolve_features(historical, selected_features, USE_LIVE_DATA)

# The newest 20% is never seen by the search, so it gives an unbiased final score.
split = time_holdout_split(historical, features)
X_search, X_hold, y_search, y_hold = split

configured = dict(params["model"]["xgboost"])

search = BayesSearchCV(
    RecencyXGB(features=features, base_params=configured),
    {
        "max_depth": Integer(2, 5),
        "eta": Real(0.005, 0.15, prior="log-uniform"),
        "n_estimators": Integer(100, 600),
        "half_life_days": Real(60, 365),
        "min_child_weight": Integer(1, 20),
        "reg_lambda": Real(0.1, 50, prior="log-uniform"),
        "subsample": Real(0.5, 1.0),
        "colsample_bytree": Real(0.5, 1.0),
    },
    n_iter=50,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="roc_auc",
    random_state=42,
    n_jobs=1,
    verbose=0,
)
search.fit(X_search, y_search)

best_params = {k: v.item() if hasattr(v, "item") else v for k, v in search.best_params_.items()}
half_life_days = best_params.pop("half_life_days")
merged_xgb_params = {**configured, **best_params}

assert half_life_days > 0
print(f"Best CV ROC AUC: {search.best_score_:.3f}")
print(f"recency_half_life_days: {half_life_days:.0f}")
print("xgboost:")
for p in merged_xgb_params:
    print(f"  {p}: {merged_xgb_params[p]}")


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

In [ ]:
hold_df = X_hold.assign(will_train_tomorrow=y_hold)
cols = ["auc", "accuracy", "baseline_accuracy"]
pd.DataFrame({
    "before": walk_forward_evaluate(hold_df, xgb_params=configured, features=features).mean()[cols],
    "after": walk_forward_evaluate(hold_df, xgb_params=merged_xgb_params, features=features).mean()[cols],
})


,before,after
auc,0.689032,0.698242
accuracy,0.709091,0.718182
baseline_accuracy,0.700000,0.700000


In [ ]:
before = holdout_report(split, features, configured, params["model"]["recency_half_life_days"])
after = holdout_report(split, features, merged_xgb_params, half_life_days)
pd.DataFrame({"before": before, "after": after})


,before,after
train_auc,0.857691,0.912688
holdout_auc,0.600122,0.603378


In [ ]:
anti_overfit = {
    "shallow_stumps": {"max_depth": 2, "eta": 0.02, "n_estimators": 200, "min_child_weight": 10,
                       "reg_lambda": 10, "subsample": 0.7, "colsample_bytree": 0.7},
    "very_regularised": {"max_depth": 3, "eta": 0.01, "n_estimators": 300, "min_child_weight": 20,
                         "reg_lambda": 30, "reg_alpha": 1, "subsample": 0.6, "colsample_bytree": 0.5, "gamma": 1},
    "tiny_slow": {"max_depth": 2, "eta": 0.01, "n_estimators": 150, "min_child_weight": 15,
                  "reg_lambda": 20, "subsample": 0.5, "colsample_bytree": 0.5},
    "depth1_linear_ish": {"max_depth": 1, "eta": 0.05, "n_estimators": 200, "min_child_weight": 10,
                          "reg_lambda": 10, "subsample": 0.7, "colsample_bytree": 0.7},
}
half_life = params["model"]["recency_half_life_days"]

pd.DataFrame({
    name: holdout_report(split, features, {**configured, **p}, half_life)
    for name, p in {"baseline": {}, **anti_overfit}.items()
})

,baseline,shallow_stumps,very_regularised,tiny_slow,depth1_linear_ish
train_auc,0.857691,0.811715,0.723574,0.765029,0.799056
holdout_auc,0.600122,0.656695,0.589744,0.625204,0.643569


In [ ]:
pd.DataFrame({
    "half_life=180": holdout_report(split, features, configured, 180),
    "half_life=365": holdout_report(split, features, configured, 365),
    "no weighting": holdout_report(split, features, configured, float("inf")),
})


,half_life=180,half_life=365,no weighting
train_auc,0.857691,0.892566,0.901750
holdout_auc,0.600122,0.578653,0.606736


In [ ]:
if NORI_AVAILABLE:
    # Place the code that should run when NORI is available here
    # currently can only run in dev container cus computer too old
    # but computer also too old to run dev container it seems
    # but id like to compare it to my tuned XGBoost model
    pass